# 03 · Crear la base de datos en MySQL

Crea la base de datos `nba_triples`, las tablas de `sql/01_esquema.sql` y carga los CSV de `data/clean`.

Antes de ejecutar:
1. Ejecuta `02_limpieza.ipynb`.
2. Copia `.env.example` como `.env` en la raíz del repo y pon tu usuario y contraseña de MySQL.

In [1]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# lee usuario y contraseña del archivo .env de la raíz del repo
load_dotenv("../.env")

DB_NAME = os.getenv("DB_NAME", "nba_triples")

def crear_url(base_de_datos=None):
    return URL.create(
        "mysql+pymysql",
        username=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST", "localhost"),
        port=int(os.getenv("DB_PORT", 3306)),
        database=base_de_datos,
        query={"charset": "utf8mb4"},
    )

## 1. Crear la base de datos

In [2]:
engine_servidor = create_engine(crear_url())
with engine_servidor.begin() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} CHARACTER SET utf8mb4"))

engine = create_engine(crear_url(DB_NAME))
print("Conectado a", DB_NAME)

Conectado a nba_triples


## 2. Crear las tablas con sql/01_esquema.sql

In [3]:
esquema = Path("../sql/01_esquema.sql").read_text(encoding="utf-8")

# quitamos los comentarios y ejecutamos las sentencias una a una
lineas = [l for l in esquema.splitlines() if not l.strip().startswith("--")]
sentencias = [s.strip() for s in "\n".join(lineas).split(";") if s.strip()]

with engine.begin() as conn:
    for sentencia in sentencias:
        conn.execute(text(sentencia))

with engine.connect() as conn:
    print(pd.read_sql("SHOW TABLES", conn))

  Tables_in_nba_triples
0               equipos
1          estadisticas
2             jugadores


## 3. Cargar los datos

El orden importa: primero `jugadores` y `equipos`, y después `estadisticas`, que apunta a las otras dos con claves foráneas.

In [4]:
CLEAN_DIR = Path("../data/clean")

for tabla in ["jugadores", "equipos", "estadisticas"]:
    df = pd.read_csv(CLEAN_DIR / f"{tabla}.csv")
    df.to_sql(tabla, engine, if_exists="append", index=False, chunksize=2000)
    print(tabla, "->", len(df), "filas cargadas")

jugadores -> 3870 filas cargadas


equipos -> 1344 filas cargadas


estadisticas -> 23472 filas cargadas


## 4. Comprobación

In [5]:
with engine.connect() as conn:
    for tabla in ["jugadores", "equipos", "estadisticas"]:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {tabla}")).scalar()
        print(tabla, n)

jugadores 3870
equipos 1344
estadisticas 23472
